# **Project Name**  - Vaccination Data Analysis and Visualization

##### **Project Type**   - Python,SQL,EDA,Power BI
##### **Contribution**   - Individual
##### **Team Member 1 -**  Gade Pavan Kumar Reddy

# **Project Summary -**

This project centers on analyzing global vaccination and infectious disease data to support better public health decisions. By designing and implementing a structured MySQL database, we integrated diverse datasets covering vaccination coverage, disease incidence, vaccine schedules, and country-level details. The cleaned and transformed data—comprising hundreds of thousands of records—are loaded into a relational database and connected to Power BI for advanced analytics and visualization.

Using Power BI, we developed interactive dashboards that enable users to explore trends in vaccination rates and disease incidence, uncover regional disparities, and evaluate the effectiveness of immunization programs. Scatter plots reveal the relationship between vaccination coverage and disease reduction, while geographical heatmaps identify regions of concern and progress. Trend lines, KPI indicators, and slicers facilitate dynamic filtering by year, country, or disease, making the analysis accessible and actionable for health officials and policymakers.

The solution supports scenario-based investigations—such as tracking the impact of new vaccine introductions, identifying regions for resource allocation, and monitoring progress toward international immunization targets. The system is set up for scheduled refreshes, ensuring that stakeholders always work with up-to-date information. This project offers a scalable and adaptable blueprint for turning raw health data into actionable insights, empowering public health interventions and resource prioritization for maximum impact.

# **GitHub Link**

Link- https://github.com/pavangade31/Vaccination-data-and-visualization-project

# **Problem Statement**

Despite ongoing vaccination efforts, many regions face persistent challenges in achieving high immunization coverage and controlling vaccine-preventable diseases. Fragmented data sources and a lack of integrated analytical tools make it difficult for health agencies to identify coverage gaps, evaluate program effectiveness, and respond rapidly to emerging outbreaks. Without a centralized, real-time analysis platform, decision-makers struggle to understand where interventions are most needed and to monitor progress toward health objectives.

This project addresses the need for an end-to-end data pipeline and interactive analytics—transforming disparate vaccination and disease data into unified, actionable dashboards to enable more targeted, effective public health decisions and interventions.

# ***Let's Begin !***

## Step 1: Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine,text
import warnings
warnings.filterwarnings('ignore')

## Step 2: Load Excel Data

In [2]:
coverage_df = pd.read_excel('coverage-data.xlsx')                  
incidence_df = pd.read_excel('incidence-rate-data.xlsx')                
cases_df = pd.read_excel('reported-cases-data.xlsx')                    
intro_df = pd.read_excel('vaccine-introduction-data.xlsx')              
schedule_df = pd.read_excel('vaccine-schedule-data.xlsx') 

## Step 3: Data Cleaning and Imputation

### 3.1: Checking Shape of all dataframes

In [3]:
dfs = {
    "coverage_df": coverage_df,
    "incidence_df": incidence_df,
    "cases_df": cases_df,
    "intro_df": intro_df,
    "schedule_df": schedule_df
}

for name, df in dfs.items():
    print(f"Shape of {name}:")
    print(df.shape)
    print("---------------------------------------")

Shape of coverage_df:
(399859, 11)
---------------------------------------
Shape of incidence_df:
(84946, 8)
---------------------------------------
Shape of cases_df:
(84870, 7)
---------------------------------------
Shape of intro_df:
(138321, 6)
---------------------------------------
Shape of schedule_df:
(8053, 12)
---------------------------------------


### 3.2: Checking Percentage of Missing Values in all dataframes

In [4]:
for name, df in dfs.items():
    print(f"Missing values percentage for every column in {name}:")
    print(df.isna().mean()*100)
    print("---------------------------------------")

Missing values percentage for every column in coverage_df:
GROUP                             0.000000
CODE                              0.000250
NAME                              0.318862
YEAR                              0.000250
ANTIGEN                           0.000250
ANTIGEN_DESCRIPTION               0.000250
COVERAGE_CATEGORY                 0.000250
COVERAGE_CATEGORY_DESCRIPTION     0.000250
TARGET_NUMBER                    80.235533
DOSES                            80.161257
COVERAGE                         42.360432
dtype: float64
---------------------------------------
Missing values percentage for every column in incidence_df:
GROUP                   0.000000
CODE                    0.001177
NAME                    0.001177
YEAR                    0.001177
DISEASE                 0.001177
DISEASE_DESCRIPTION     0.001177
DENOMINATOR             0.001177
INCIDENCE_RATE         27.502178
dtype: float64
---------------------------------------
Missing values percentage for ever

### 3.3: Cleaning and Imputing Coverage Data

In [40]:
df = coverage_df.copy()
df = df.drop(columns=['TARGET_NUMBER', 'DOSES'])

# Impute 'NAME'
df['NAME'] = df.groupby('CODE')['NAME'].transform(lambda x: x.ffill().bfill())
df['NAME'].fillna('Unknown', inplace=True)

# Function to impute with median only if median > 0, else keep NaN for second fill
def impute_nonzero_median(x):
    med = x.median()
    if med > 0:
        return x.fillna(med)
    else:
        return x

df['COVERAGE'] = df.groupby(['CODE', 'ANTIGEN'])['COVERAGE'].transform(impute_nonzero_median)

# Fill remaining NaNs with overall median excluding zeros
overall_median = df.loc[df['COVERAGE'] > 0, 'COVERAGE'].median()
df['COVERAGE'] = df['COVERAGE'].fillna(overall_median)

# Drop rows with any remaining missing data
df.dropna(inplace=True)

clean_coverage_df= df.copy()

**Remove**: **TARGET_NUMBER** (80.24% missing) and **DOSES** (80.16% missing) – too much missingness for reliable imputation.

Imputete: NAME (0.32%) with forward-fill/backward-fill by 'CODE'; COVERAGE (42.36%) with median by group (still valuable for correlation/trend
).

All other columns have negligible missingness—fill or drop remaining NaNs.

### 3.4: Cleaning and Imputing Incidence Data

In [43]:
# --- Cleaning incidence_df ---
df_incidence = incidence_df.copy()

# Fill 'NAME' missing by forward/backward fill within 'CODE'
df_incidence['NAME'] = df_incidence.groupby('CODE')['NAME'].transform(lambda x: x.ffill().bfill())
df_incidence['NAME'].fillna('Unknown', inplace=True)

# Function to impute incidence_rate only if group's median > 0
def impute_nonzero_median(x):
    med = x.median()
    if med > 0:
        return x.fillna(med)
    else:
        return x

df_incidence['INCIDENCE_RATE'] = df_incidence.groupby('DISEASE')['INCIDENCE_RATE'].transform(impute_nonzero_median)

# Fill remaining NaNs with overall median excluding zeros
overall_median_incidence = df_incidence.loc[df_incidence['INCIDENCE_RATE'] > 0, 'INCIDENCE_RATE'].median()
df_incidence['INCIDENCE_RATE'] = df_incidence['INCIDENCE_RATE'].fillna(overall_median_incidence)

# Drop any rows still having missing values
df_incidence.dropna(inplace=True)

clean_incidence_df = df_incidence.copy()

**Remove**: None—the only column with significant missingness is INCIDENCE_RATE (27.50%), still useful for analysis.


Impute: INCIDENCE_RATE by median per 'DISEASE'; other columns use forward-fill/backward-fill by 'CODE'.

### 3.5: Cleaning and Imputing Cases Data

In [46]:
# --- Cleaning cases_df ---
df_cases = cases_df.copy()

# Fill 'NAME' missing by forward/backward fill within 'CODE'
df_cases['NAME'] = df_cases.groupby('CODE')['NAME'].transform(lambda x: x.ffill().bfill())
df_cases['NAME'].fillna('Unknown', inplace=True)

# Impute CASES only if group's median > 0
df_cases['CASES'] = df_cases.groupby('DISEASE')['CASES'].transform(impute_nonzero_median)

# Fill remaining NaNs with overall median excluding zeros
overall_median_cases = df_cases.loc[df_cases['CASES'] > 0, 'CASES'].median()
df_cases['CASES'] = df_cases['CASES'].fillna(overall_median_cases)

# Drop rows with missing data
df_cases.dropna(inplace=True)

clean_cases_df = df_cases.copy()

**Remove**: None—CASES missingness (22.86%) is high, but still usable for reduction analysis.

Impute: CASES by median per 'DISEASE'; fill others as above.

### 3.6: Cleaning and Imputing Vaccine Introduction Data

In [49]:
df = intro_df.copy()
for col in ['COUNTRYNAME', 'WHO_REGION', 'YEAR', 'DESCRIPTION', 'INTRO']:
    df[col] = df[col].ffill().bfill()
df.dropna(inplace=True)
clean_intro_df = df.copy()

**Remove/Impute**: Very low missingness (<1%); forward-fill/backward-fill all missing categorical columns (by 'ISO_3_CODE' or globally). Drop if any missing remain.

### 3.7: Cleaning and Imputing Schedule Data

In [52]:
df = schedule_df.copy()
df = df.drop(columns=['TARGETPOP'])  # remove due to high missingness

# Fill others by group
for col in ['COUNTRYNAME', 'WHO_REGION', 'YEAR', 'VACCINECODE', 'VACCINE_DESCRIPTION', 'SCHEDULEROUNDS', 'TARGETPOP_DESCRIPTION', 'GEOAREA']:
    df[col] = df.groupby('ISO_3_CODE')[col].transform(lambda x: x.ffill().bfill())
# Impute AGEADMINISTERED and SOURCECOMMENT
df['AGEADMINISTERED'].fillna('Not specified', inplace=True)
df['SOURCECOMMENT'].fillna('No comment', inplace=True)
df.dropna(inplace=True)
clean_schedule_df = df.copy()

**Remove**: TARGETPOP (52.87%)—over half missing, drop for robust analysis.

Impute: AGEADMINISTERED (12.99%) and SOURCECOMMENT (36.19%)—impute missing values with placeholders; other columns impute by ffill/bfill by 'ISO_3_CODE'.

### 3.8: Checking the shape of Cleaned Dataframes

In [55]:
dfs_new = {
    "coverage_df": clean_coverage_df,
    "incidence_df": clean_incidence_df,
    "cases_df": clean_cases_df,
    "intro_df": clean_intro_df,
    "schedule_df": clean_schedule_df
}

for name, df in dfs_new.items():
    print(f"Shape of {name} after cleaning and imputation:")
    print(df.shape)
    print("---------------------------------------")

Shape of coverage_df after cleaning and imputation:
(399858, 9)
---------------------------------------
Shape of incidence_df after cleaning and imputation:
(84945, 8)
---------------------------------------
Shape of cases_df after cleaning and imputation:
(84869, 7)
---------------------------------------
Shape of intro_df after cleaning and imputation:
(138321, 6)
---------------------------------------
Shape of schedule_df after cleaning and imputation:
(8052, 11)
---------------------------------------


### 3.9: Checking Percentage of Missing Values in all Cleaned dataframes

In [56]:
for name, df in dfs_new.items():
    print(f"Missing values percentage for every column in {name} after cleaning and imputation:")
    print(df.isna().mean()*100)
    print("---------------------------------------")

Missing values percentage for every column in coverage_df after cleaning and imputation:
GROUP                            0.0
CODE                             0.0
NAME                             0.0
YEAR                             0.0
ANTIGEN                          0.0
ANTIGEN_DESCRIPTION              0.0
COVERAGE_CATEGORY                0.0
COVERAGE_CATEGORY_DESCRIPTION    0.0
COVERAGE                         0.0
dtype: float64
---------------------------------------
Missing values percentage for every column in incidence_df after cleaning and imputation:
GROUP                  0.0
CODE                   0.0
NAME                   0.0
YEAR                   0.0
DISEASE                0.0
DISEASE_DESCRIPTION    0.0
DENOMINATOR            0.0
INCIDENCE_RATE         0.0
dtype: float64
---------------------------------------
Missing values percentage for every column in cases_df after cleaning and imputation:
GROUP                  0.0
CODE                   0.0
NAME                  

### 3.10: Checking Datatypes of columns in all Cleaned dataframes

In [57]:
for name, df in dfs_new.items():
    print(f"Data types of every column in {name} after cleaning and imputation:")
    print(df.dtypes)
    print("---------------------------------------")

Data types of every column in coverage_df after cleaning and imputation:
GROUP                             object
CODE                              object
NAME                              object
YEAR                             float64
ANTIGEN                           object
ANTIGEN_DESCRIPTION               object
COVERAGE_CATEGORY                 object
COVERAGE_CATEGORY_DESCRIPTION     object
COVERAGE                         float64
dtype: object
---------------------------------------
Data types of every column in incidence_df after cleaning and imputation:
GROUP                   object
CODE                    object
NAME                    object
YEAR                   float64
DISEASE                 object
DISEASE_DESCRIPTION     object
DENOMINATOR             object
INCIDENCE_RATE         float64
dtype: object
---------------------------------------
Data types of every column in cases_df after cleaning and imputation:
GROUP                   object
CODE                    obj

### 3.11: Changing Float values to Integer 

In [58]:
clean_coverage_df['YEAR'] = clean_coverage_df['YEAR'].astype(int)
clean_coverage_df['COVERAGE'] = clean_coverage_df['COVERAGE'].astype(int)
clean_incidence_df['YEAR'] = clean_incidence_df['YEAR'].astype(int)
clean_incidence_df['INCIDENCE_RATE'] = clean_incidence_df['INCIDENCE_RATE'].astype(int)
clean_cases_df['YEAR'] = clean_cases_df['YEAR'].astype(int)
clean_cases_df['CASES'] = clean_cases_df['CASES'].astype(int)
clean_intro_df['YEAR'] = clean_intro_df['YEAR'].astype(int)
clean_schedule_df['YEAR'] = clean_schedule_df['YEAR'].astype(int)
clean_schedule_df['SCHEDULEROUNDS'] = clean_schedule_df['SCHEDULEROUNDS'].astype(int)

### 3.12: Saving all cleaned dataframes into Excel files

In [ ]:
clean_coverage_df.to_excel("clean_coverage_data.xlsx", index=False)
clean_incidence_df.to_excel("clean_incidence_rate_data.xlsx", index=False)
clean_cases_df.to_excel("clean_reported_cases_data.xlsx", index=False)
clean_intro_df.to_excel("clean_vaccine_introduction_data.xlsx", index=False)
clean_schedule_df.to_excel("clean_vaccine_schedule_data.xlsx", index=False)

## Step 4: Connecting to MySQL Database and loading data

### 4.1: Creating Database

In [36]:
import pymysql

host = "localhost"
user = "root"
password = "root"

try:
    conn = pymysql.connect(
        host=host,
        user=user,
        password=password
    )
    cursor = conn.cursor()

    
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS vaccination_data_analysis")
    conn.commit()
    print("Database created successfully.")

except pymysql.Error as e:
    print(f"Error creating database: {e}")
finally:
    conn.close()

Database created successfully.


### 4.2: Creating Tables

In [37]:
host = "localhost"
user = "root"
password = "root"
database = "vaccination_data_analysis"

try:
    conn = pymysql.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = conn.cursor()

    # Table 1: Coverage Data
    create_coverage_table = """
    CREATE TABLE Coverage (
        `GROUP` VARCHAR(255),
        CODE VARCHAR(100),
        NAME VARCHAR(255),
        YEAR INT,
        ANTIGEN VARCHAR(255),
        ANTIGEN_DESCRIPTION TEXT,
        COVERAGE_CATEGORY VARCHAR(255),
        COVERAGE_CATEGORY_DESCRIPTION TEXT,
        COVERAGE FLOAT
    );
    """
    cursor.execute(create_coverage_table)

    # Table 2: Incidence Rate
    create_incidence_rate_table = """
    CREATE TABLE IncidenceRate (
        `GROUP` VARCHAR(255),
        CODE VARCHAR(100),
        NAME VARCHAR(255),
        YEAR INT,
        DISEASE VARCHAR(255),
        DISEASE_DESCRIPTION TEXT,
        DENOMINATOR VARCHAR(255),
        INCIDENCE_RATE FLOAT
    );
    """
    cursor.execute(create_incidence_rate_table)

    # Table 3: Reported Cases
    create_reported_cases_table = """
    CREATE TABLE ReportedCases (
        `GROUP` VARCHAR(255),
        CODE VARCHAR(100),
        NAME VARCHAR(255),
        YEAR INT,
        DISEASE VARCHAR(255),
        DISEASE_DESCRIPTION TEXT,
        CASES INT
    );
    """
    cursor.execute(create_reported_cases_table)

    # Table 4: Vaccine Introduction
    create_vaccine_introduction_table = """
    CREATE TABLE VaccineIntroduction (
        ISO_3_CODE VARCHAR(100),
        COUNTRYNAME VARCHAR(255),
        WHO_REGION VARCHAR(255),
        YEAR INT,
        DESCRIPTION TEXT,
        INTRO VARCHAR(100)
    );
    """
    cursor.execute(create_vaccine_introduction_table)

    # Table 5: Vaccine Schedule
    create_vaccine_schedule_table = """
    CREATE TABLE VaccineSchedule (
        ISO_3_CODE VARCHAR(100),
        COUNTRYNAME VARCHAR(255),
        WHO_REGION VARCHAR(255),
        YEAR INT,
        VACCINECODE VARCHAR(255),
        VACCINE_DESCRIPTION TEXT,
        SCHEDULEROUNDS INT,
        TARGETPOP_DESCRIPTION TEXT,
        GEOAREA VARCHAR(100),
        AGEADMINISTERED VARCHAR(255),
        SOURCECOMMENT TEXT
    );
    """
    cursor.execute(create_vaccine_schedule_table)

    conn.commit()
    print("Tables created successfully.")

except pymysql.Error as e:
    print(f"Error creating tables: {e}")

finally:
    conn.close()

Tables created successfully.


### 4.3: Importing Data from excel

In [62]:
host = "localhost"
user = "root"
password = "root"
database = "vaccination_data_analysis"

def import_data_from_excel(file_path, table_name):
    try:
        conn = pymysql.connect(
            host=host,
            user=user,
            password=password,
            database=database
        )
        cursor = conn.cursor()

        df = pd.read_excel(file_path)
        for index, row in df.iterrows():
            values = tuple(row)
            sql = f"INSERT INTO {table_name} VALUES {values}"
            cursor.execute(sql)
        conn.commit()
        print(f"Data imported successfully from {file_path} to {table_name}")

    except Exception as e:
        print(f"Error importing data from {file_path}: {e}")

    finally:
        conn.close()

# file path
coverage_file = "clean_coverage_data.xlsx"
incidence_rate_file = "clean_incidence_rate_data.xlsx"
reported_cases_file = "clean_reported_cases_data.xlsx"
vaccine_introduction_file = "clean_vaccine_introduction_data.xlsx"
vaccine_schedule_file = "clean_vaccine_schedule_data.xlsx"

import_data_from_excel(coverage_file, "Coverage")
import_data_from_excel(incidence_rate_file, "IncidenceRate")
import_data_from_excel(reported_cases_file, "ReportedCases")
import_data_from_excel(vaccine_introduction_file, "VaccineIntroduction")
import_data_from_excel(vaccine_schedule_file, "VaccineSchedule")

Data imported successfully from clean_coverage_data.xlsx to Coverage
Data imported successfully from clean_incidence_rate_data.xlsx to IncidenceRate
Data imported successfully from clean_reported_cases_data.xlsx to ReportedCases
Data imported successfully from clean_vaccine_introduction_data.xlsx to VaccineIntroduction
Data imported successfully from clean_vaccine_schedule_data.xlsx to VaccineSchedule


### 4.4: Normalization

To creating separate tables for countries, diseases, years, WHO_Region to avoid redundancy and improve querying performance.

In [63]:
host = "localhost"
user = "root"
password = "root"
database = "vaccination_data_analysis"

try:
    conn = pymysql.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = conn.cursor()

    # Creating Countries table
    create_countries_table = """
    CREATE TABLE Countries (
        `GROUP` VARCHAR(255),
        CODE VARCHAR(100),
        NAME VARCHAR(255),
        PRIMARY KEY (CODE) 
    );
    """
    cursor.execute(create_countries_table)

    # Insert unique country data from Coverage table
    insert_countries_from_coverage = """
    INSERT IGNORE INTO Countries (`GROUP`, CODE, NAME)
    SELECT DISTINCT `GROUP`, CODE, NAME 
    FROM Coverage;
    """
    cursor.execute(insert_countries_from_coverage)

    conn.commit()
    print("Countries table populated successfully.")

except pymysql.Error as e:
    print(f"Error populating Countries table: {e}")

finally:
    conn.close()

Countries table populated successfully.


In [64]:
host = "localhost"
user = "root"
password = "root"
database = "vaccination_data_analysis"

try:
    conn = pymysql.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = conn.cursor()

    # Create Years table
    create_years_table = """
    CREATE TABLE Years (
        YEAR INT PRIMARY KEY
    );
    """
    cursor.execute(create_years_table)

    # Insert unique years from vaccineintroduction table
    insert_years_from_vaccineintroduction = """
    INSERT IGNORE INTO Years (YEAR)
    SELECT DISTINCT YEAR 
    FROM VaccineIntroduction;
    """
    cursor.execute(insert_years_from_vaccineintroduction)

    conn.commit()
    print("Years table populated successfully.")

except pymysql.Error as e:
    print(f"Error populating Years table: {e}")

finally:
    conn.close()

Years table populated successfully.


In [65]:
host = "localhost"
user = "root"
password = "root"
database = "vaccination_data_analysis"

try:
    conn = pymysql.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = conn.cursor()

    # Create Diseases table
    create_diseases_table = """
    CREATE TABLE Diseases (
        DISEASE VARCHAR(255) PRIMARY KEY,
        DISEASE_DESCRIPTION TEXT
    );
    """
    cursor.execute(create_diseases_table)

    # Insert unique diseases from IncidenceRate table
    insert_diseases_from_incidencerate = """
    INSERT IGNORE INTO Diseases (DISEASE, DISEASE_DESCRIPTION)
    SELECT DISTINCT DISEASE, DISEASE_DESCRIPTION 
    FROM IncidenceRate;
    """
    cursor.execute(insert_diseases_from_incidencerate)

    conn.commit()
    print("Diseases table populated successfully.")

except pymysql.Error as e:
    print(f"Error populating Diseases table: {e}")

finally:
    conn.close()

Diseases table populated successfully.


In [74]:
host = "localhost"
user = "root"
password = "root"
database = "vaccination_data_analysis"

try:
    conn = pymysql.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = conn.cursor()

    # Create WHO_Region table
    create_who_region_table = """
    CREATE TABLE WHO_Region (
        ISO_3_CODE VARCHAR(100),
        COUNTRYNAME VARCHAR(255),
        WHO_REGION VARCHAR(255),
        PRIMARY KEY (ISO_3_CODE, COUNTRYNAME) 
    );
    """
    cursor.execute(create_who_region_table)

    # Insert unique WHO_Region data from VaccineSchedule table
    insert_who_region_from_VaccineSchedule = """
    INSERT IGNORE INTO WHO_Region (ISO_3_CODE, COUNTRYNAME, WHO_REGION)
    SELECT DISTINCT ISO_3_CODE, COUNTRYNAME, WHO_REGION FROM VaccineIntroduction
    UNION
    SELECT DISTINCT ISO_3_CODE, COUNTRYNAME, WHO_REGION FROM VaccineSchedule;
    """
    cursor.execute(insert_who_region_from_VaccineSchedule)

    conn.commit()
    print("WHO_Region table populated successfully.")

except pymysql.Error as e:
    print(f"Error populating WHO_Region table: {e}")

finally:
    conn.close()

WHO_Region table populated successfully.


In [75]:
host = "localhost"
user = "root"
password = "root"
database = "vaccination_data_analysis"

try:
    conn = pymysql.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = conn.cursor()
    
    # Add primary and foreign keys to Coverage table
    alter_coverage_table = """
    ALTER TABLE Coverage
    ADD FOREIGN KEY (CODE) REFERENCES Countries(CODE), 
    ADD FOREIGN KEY (YEAR) REFERENCES Years(YEAR);
    """
    cursor.execute(alter_coverage_table)

    # Add primary and foreign keys to IncidenceRate table
    alter_incidencerate_table = """
    ALTER TABLE IncidenceRate
    ADD FOREIGN KEY (CODE) REFERENCES Countries(CODE),
    ADD FOREIGN KEY (DISEASE) REFERENCES Diseases(DISEASE),
    ADD FOREIGN KEY (YEAR) REFERENCES Years(YEAR);
    """
    cursor.execute(alter_incidencerate_table)

    # Add primary and foreign keys to ReportedCases table
    alter_reportedcases_table = """
    ALTER TABLE ReportedCases
    ADD FOREIGN KEY (CODE) REFERENCES Countries(CODE),
    ADD FOREIGN KEY (DISEASE) REFERENCES Diseases(DISEASE),
    ADD FOREIGN KEY (YEAR) REFERENCES Years(YEAR);
    """
    cursor.execute(alter_reportedcases_table)

    # Add primary and foreign keys to VaccineIntroduction table
    alter_vaccineintroduction_table = """
    ALTER TABLE VaccineIntroduction
    ADD FOREIGN KEY (ISO_3_CODE, COUNTRYNAME) REFERENCES WHO_Region(ISO_3_CODE, COUNTRYNAME), 
    ADD FOREIGN KEY (YEAR) REFERENCES Years(YEAR); 
    """
    cursor.execute(alter_vaccineintroduction_table)

    # Add primary and foreign keys to VaccineSchedule table
    alter_vaccineschedule_table = """
    ALTER TABLE VaccineSchedule
    ADD FOREIGN KEY (ISO_3_CODE, COUNTRYNAME) REFERENCES WHO_Region(ISO_3_CODE, COUNTRYNAME), 
    ADD FOREIGN KEY (YEAR) REFERENCES Years(YEAR); 
    """
    cursor.execute(alter_vaccineschedule_table)

    conn.commit()
    print("Primary and foreign keys added successfully.")

except pymysql.Error as e:
    print(f"Error adding primary and foreign keys: {e}")

finally:
    conn.close()

Primary and foreign keys added successfully.


### 4.5 Dropping Duplicate Columns from the Main Tables, as Foreign Key has been established

In [76]:
host = "localhost"
user = "root"
password = "root"
database = "vaccination_data_analysis"

try:
    conn = pymysql.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = conn.cursor()
    
    
    drop_columns_queries = [
        # Drop duplicate columns from Coverage
        "ALTER TABLE Coverage DROP COLUMN NAME, DROP COLUMN `GROUP`;",
        # Drop duplicate columns from IncidenceRate
        "ALTER TABLE IncidenceRate DROP COLUMN NAME, DROP COLUMN `GROUP`;",
        "ALTER TABLE IncidenceRate DROP COLUMN DISEASE_DESCRIPTION;",
        # Drop duplicate columns from ReportedCases
        "ALTER TABLE ReportedCases DROP COLUMN NAME, DROP COLUMN `GROUP`;",
        "ALTER TABLE ReportedCases DROP COLUMN DISEASE_DESCRIPTION;",
        # Drop duplicate columns from VaccineIntroduction
        "ALTER TABLE VaccineIntroduction DROP COLUMN COUNTRYNAME, DROP COLUMN WHO_REGION;",
        # Drop duplicate columns from VaccineSchedule
        "ALTER TABLE VaccineSchedule DROP COLUMN COUNTRYNAME, DROP COLUMN WHO_REGION;"
    ]
    for query in drop_columns_queries:
        try:
            cursor.execute(query)
            print(f"Executed: {query.strip()}")
        except pymysql.Error as e:
            print(f"Skipping column drop: {query.strip()} - Error: {e}")

    conn.commit()
    print("Duplicate columns dropped successfully where applicable.")

except pymysql.Error as e:
    print(f"Error processing the database: {e}")

finally:
    conn.close()

Executed: ALTER TABLE Coverage DROP COLUMN NAME, DROP COLUMN `GROUP`;
Executed: ALTER TABLE IncidenceRate DROP COLUMN NAME, DROP COLUMN `GROUP`;
Executed: ALTER TABLE IncidenceRate DROP COLUMN DISEASE_DESCRIPTION;
Executed: ALTER TABLE ReportedCases DROP COLUMN NAME, DROP COLUMN `GROUP`;
Executed: ALTER TABLE ReportedCases DROP COLUMN DISEASE_DESCRIPTION;
Skipping column drop: ALTER TABLE VaccineIntroduction DROP COLUMN COUNTRYNAME, DROP COLUMN WHO_REGION; - Error: (1828, "Cannot drop column 'COUNTRYNAME': needed in a foreign key constraint 'vaccineintroduction_ibfk_3'")
Skipping column drop: ALTER TABLE VaccineSchedule DROP COLUMN COUNTRYNAME, DROP COLUMN WHO_REGION; - Error: (1828, "Cannot drop column 'COUNTRYNAME': needed in a foreign key constraint 'vaccineschedule_ibfk_1'")
Duplicate columns dropped successfully where applicable.


In [77]:
try:
    conn = pymysql.connect(
        host=host,
        user=user,
        password=password,
        database=database
    )
    cursor = conn.cursor()
    
    
    drop_columns_queries = [
        # Drop duplicate columns from VaccineIntroduction
        "ALTER TABLE VaccineIntroduction DROP COLUMN WHO_REGION;",
        # Drop duplicate columns from VaccineSchedule
        "ALTER TABLE VaccineSchedule DROP COLUMN WHO_REGION;"
    ]
    for query in drop_columns_queries:
        try:
            cursor.execute(query)
            print(f"Executed: {query.strip()}")
        except pymysql.Error as e:
            print(f"Skipping column drop: {query.strip()} - Error: {e}")

    conn.commit()
    print("Duplicate columns dropped successfully where applicable.")

except pymysql.Error as e:
    print(f"Error processing the database: {e}")

finally:
    conn.close()

Executed: ALTER TABLE VaccineIntroduction DROP COLUMN WHO_REGION;
Executed: ALTER TABLE VaccineSchedule DROP COLUMN WHO_REGION;
Duplicate columns dropped successfully where applicable.
